# MetaJudge: Evidence-Assisted Self-Correction (C2)

Two-turn protocol: model answers, then receives a reviewer's note with external evidence
that may be accurate, misleading, or partially relevant. Tests whether external evidence
helps or damages performance. 23 items including deceptive traps.

**Metacognitive Control** · Nelson & Narens (1990) · v6.1


In [ ]:
import datetime
import sys, os, json
sys.path.insert(0, "/kaggle/input/datasets/seanmcgee2025/metajudge-package-v6-1")
DATA_ROOT = "/kaggle/input/datasets/seanmcgee2025/metajudge-data-v6-1"

import kaggle_benchmarks as kbench
from kaggle_benchmarks import chats
from metajudge.scoring.grading_v2 import grade_item, load_registry
from metajudge.tasks.self_correction_v2 import (
    T1_SUFFIX, C2_T2_TEMPLATE, parse_answer_confidence,
    compute_edit_similarity, resolve_t2_answer,
)
from metajudge.scoring.self_correction_v2 import classify_transition

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
import numpy as np

ANCHOR_C2_FLOOR = -0.2   # Empirical: min - 2*SD across all model-runs
ANCHOR_C2_CEIL  = 0.2   # Empirical: max + 2*SD across all model-runs

def normalize(score, floor, ceil):
    return max(0.0, min(1.0, (score - floor) / (ceil - floor)))

print(f"Scoring: transition-based (package), anchor C2=[{ANCHOR_C2_FLOOR}, {ANCHOR_C2_CEIL}]")


In [ ]:
# (no schema needed — uses text parsing)


In [ ]:
import pandas as pd

with open(os.path.join(DATA_ROOT, "family_c_items.json")) as f:
    all_fc_items = json.load(f)

with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

fc_excluded = set(manifest.get("family_c", {}).get("excluded_items", []))
fc_items = [it for it in all_fc_items if it["item_id"] not in fc_excluded]
c2_items = [it for it in fc_items if it.get("subfamily") == "C2"]
c2_df = pd.DataFrame(c2_items)

REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))


In [ ]:
@kbench.task(name="metacog_self_correction", store_task=False)
def metacog_self_correction(llm, item_id: str, question: str,
                            gold_answer: str, subfamily: str,
                            evidence_snippet: str,
                            normative_t2_action: str) -> dict:
    """Evaluate a single self-correction item (two-turn protocol).

    Turn 1: Model answers the question with ANSWER + CONFIDENCE.
    Turn 2: Model receives review prompt and may revise.
      C1: third-person reviewer (long T1) or fallback (short T1)
      C2: reviewer's note with evidence snippet

    Returns dict with item-level audit data.
    """
    with chats.new():
        # Turn 1
        t1_resp = llm.prompt(question + T1_SUFFIX)
        t1_text = str(t1_resp)
        t1_answer, t1_conf = parse_answer_confidence(t1_text)

        # Turn 2
        if subfamily == "C1":
            t2_prompt = (C1_T2_PRIMARY if len(t1_text) > C1_PRIMARY_MIN_LENGTH
                         else C1_T2_FALLBACK)
        else:  # C2
            t2_prompt = C2_T2_TEMPLATE.format(
                evidence=evidence_snippet or "")
        t2_resp = llm.prompt(t2_prompt)
        t2_text = str(t2_resp)
        t2_answer, t2_conf = parse_answer_confidence(t2_text)

    # Resolve T2 answer (handle confirmation-without-restatement)
    t2_answer_resolved = resolve_t2_answer(t2_answer, t1_answer, gold_answer)

    # Grade both turns
    t1_correct = grade_item(item_id, t1_answer, REGISTRY).get("correct", False)
    t2_correct = grade_item(item_id, t2_answer_resolved, REGISTRY).get("correct", False)

    # Classify transition
    revised = t1_answer.strip().lower() != t2_answer.strip().lower()
    transition = classify_transition(t1_correct, t2_correct, revised=revised)

    # Edit distance
    edit_sim = compute_edit_similarity(t1_text, t2_text)

    return {
        "item_id": item_id,
        "subfamily": subfamily,
        "gold_answer": gold_answer,
        "t1_answer": t1_answer[:200],
        "t2_answer": t2_answer[:200],
        "t1_confidence": round(t1_conf, 4) if t1_conf is not None else None,
        "t2_confidence": round(t2_conf, 4) if t2_conf is not None else None,
        "t1_correct": t1_correct,
        "t2_correct": t2_correct,
        "transition": transition,
        "t1_t2_similarity": round(edit_sim, 4),
        "normative_t2_action": normative_t2_action,
    }


In [ ]:
@kbench.task(name="metajudge_sc_c2_v61", version=6)
def metajudge_sc_c2_v61(llm) -> float:
    """Family C2 — Evidence-Assisted Self-Correction (C2) (Control).

    Returns anchor-normalized T2-T1 accuracy delta.
    Includes dual-run stochasticity check (Run 2 is display only, never blocks scoring).
    """
    model_slug = str(llm).replace("/", "_")
    sub_df = c2_df[
        ["item_id", "question", "gold_answer", "subfamily",
         "evidence_snippet", "normative_t2_action"]
    ].copy()
    sub_df["evidence_snippet"] = sub_df["evidence_snippet"].fillna("")

    # Run 1 (scored)
    with kbench.client.enable_cache():
        runs_1 = metacog_self_correction.evaluate(
            llm=[llm], evaluation_data=sub_df,
            n_jobs=4, remove_run_files=True,
            stop_condition=lambda runs: len(runs) == len(sub_df),
            max_attempts=1,
        )
    records_1 = [r.result for r in runs_1 if r.result is not None]
    n_failed_1 = len(sub_df) - len(records_1)
    if n_failed_1:
        print(f"  WARNING: {n_failed_1}/{len(sub_df)} items failed in Run 1. Score based on {len(records_1)} items.")

    # Run 2 (stochasticity check — display only, never blocks scoring)
    records_2 = []
    try:
        runs_2 = metacog_self_correction.evaluate(
            llm=[llm], evaluation_data=sub_df,
            n_jobs=4, remove_run_files=True,
            stop_condition=lambda runs: len(runs) == len(sub_df),
            max_attempts=1,
        )
        records_2 = [r.result for r in runs_2 if r.result is not None]
    except Exception as e:
        print(f"  Stochasticity Run 2 failed: {e}")

    # Score from Run 1 (always computed)
    audit = pd.DataFrame(records_1)
    t1c = audit["t1_correct"].sum()
    t2c = audit["t2_correct"].sum()
    delta = float((t2c - t1c) / len(audit))
    normalized = normalize(delta, ANCHOR_C2_FLOOR, ANCHOR_C2_CEIL)

    # Damage stats
    t1_right = audit["t1_correct"].sum()
    rw = ((audit["t1_correct"]) & (~audit["t2_correct"])).sum()
    dmg_rate = rw / t1_right if t1_right > 0 else 0.0

    print(f"C2: \u0394={delta:+.4f} Norm={normalized:.3f} Dmg={dmg_rate:.1%} [{len(audit)} items]")
    damage_items = audit[audit["t1_correct"] & ~audit["t2_correct"]]
    for _, row in damage_items.iterrows():
        print(f"    {row['item_id']}: '{row['t1_answer'][:40]}' \u2192 '{row['t2_answer'][:40]}'")

    # Stochasticity comparison (gated on sufficient Run 2 data)
    r2_lookup = {r["item_id"]: r for r in records_2}
    matched_pairs = [(r1, r2_lookup[r1["item_id"]])
                     for r1 in records_1 if r1["item_id"] in r2_lookup]
    has_stochasticity = len(matched_pairs) >= len(records_1) * 0.8

    if has_stochasticity:
        matched_r2 = [r2 for _, r2 in matched_pairs]
        t1c_2 = sum(1 for r in matched_r2 if r.get("t1_correct"))
        t2c_2 = sum(1 for r in matched_r2 if r.get("t2_correct"))
        delta_2 = float((t2c_2 - t1c_2) / len(matched_r2))
        norm_2 = normalize(delta_2, ANCHOR_C2_FLOOR, ANCHOR_C2_CEIL)

        trans_diffs = sum(1 for r1, r2 in matched_pairs
                          if r1["transition"] != r2["transition"])
        stable = len(matched_pairs) - trans_diffs

        print(f"  Stochasticity: Run 1 \u0394={delta:+.4f}, Run 2 \u0394={delta_2:+.4f} ({len(matched_pairs)}/{len(records_1)} items matched)")
        print(f"  Transition stability: {stable}/{len(matched_pairs)} items stable ({stable/len(matched_pairs):.0%})")
        print(f"  \u2192 Score ranges from {min(normalized, norm_2):.2f} to {max(normalized, norm_2):.2f} across independent runs")
    elif len(records_2) > 0:
        print(f"  Stochasticity: Run 2 incomplete ({len(matched_pairs)}/{len(records_1)} items matched). Skipping comparison.")
    else:
        print(f"  Stochasticity: insufficient Run 2 data. Displaying Run 1 only.")

    # Export (always happens)
    audit.to_csv(os.path.join(OUTPUT_DIR, f"family_c2_item_audit_{model_slug}_v6.1.csv"), index=False)
    _meta = {
        "metadata": {
            "model": str(llm), "task": "metajudge_sc_c2", "version": "v6.1",
            "timestamp": datetime.datetime.utcnow().isoformat(),
            "items_scored": len(records_1), "items_attempted": len(sub_df),
            "run_2_complete": len(records_2) == len(records_1),
            "run_2_items": len(records_2),
        },
        "run_1": records_1,
        "run_2": records_2,
    }
    with open(os.path.join(OUTPUT_DIR, f"family_c2_full_responses_{model_slug}_v6.1.json"), "w") as f:
        json.dump(_meta, f, indent=2, default=str)

    # Score from Run 1 (always returned)
    return normalized


In [ ]:
metajudge_sc_c2_v61.run(kbench.llm)
%choose metajudge_sc_c2_v61


## Methodology

Two-turn protocol: T1 answer → reviewer's note with evidence snippet → T2 revision.
Evidence may be accurate, misleading, or irrelevant — tests whether the model appropriately
integrates or rejects external information. Same transition-based scoring as C1.
Dual-run stochasticity check: Run 1 cached (scored), Run 2 independent (display only).
23 clean C2 items (evidence-assisted). Normalized using anchors [−0.20, +0.20].

For theoretical grounding, statistical methodology, and full citations,
see `docs/benchmark/v5_theoretical_backgrounder.md`.
